# Notebook 0 -- JAX/nnx warm-up (optional, ~30 min)

Ideally run this in advance of the school.
It has one job: make sure your environment works and that you can read the
JAX code in Notebooks 1-2.
This is *not* a proper JAX course -- we link out for everything generic and only
cover the small subset our notebooks actually use.

Three markers:
- ✏️ marks an exercise
- 📦 marks provided code or context (just read/run)
- ⭐ marks optional extra material

## 0. Install check -- run this first (📦)

The cell below imports every package the tutorial needs, prints their
versions, and trains a tiny model for 5 steps.
The model is the solution of the micro-exercise in section 3, inlined here
so that the cell is self-contained.

If any import fails, reinstall following the README instructions and
re-run, and bring anything you cannot resolve to the first session.
If the cell ends with `Environment OK`, you need zero setup time in Block 1.

In [ ]:
# 📦 One-cell install check.
import importlib.metadata

import jax
import jax.numpy as jnp
import numpy as np
import flax
import flax.nnx as nnx
import optax
import bijx

import iaifi_gm as gm  # helper package

print("jax     ", jax.__version__, " devices:", jax.devices())
print("flax    ", flax.__version__)
print("optax   ", optax.__version__)
print("bijx    ", bijx.__version__)
print("iaifi-gm", importlib.metadata.version("iaifi-gm"))

# 5 training steps of the section-3 linear fit, using the canonical
# train-step scaffold every tutorial notebook reuses.
rngs = nnx.Rngs(0)


class LinearModel(nnx.Module):
    def __init__(self, *, rngs: nnx.Rngs):
        self.a = nnx.Param(jax.random.normal(rngs.params()))
        self.b = nnx.Param(jnp.zeros(()))

    def __call__(self, x):
        """x: (N,) -> predictions (N,)"""
        return self.a * x + self.b


model = LinearModel(rngs=rngs)
optimizer = nnx.Optimizer(model, optax.adam(1e-1), wrt=nnx.Param)


@nnx.jit
def train_step(model, optimizer, key, batch):
    def loss_fn(model):
        x, y = batch
        return jnp.mean((model(x) - y) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads=grads, model=model)
    return loss


key = jax.random.key(0)
key_x, key_noise = jax.random.split(key)
x_data = jax.random.uniform(key_x, (256,), minval=-1.0, maxval=1.0)
y_data = 2.0 * x_data - 0.7 + 0.1 * jax.random.normal(key_noise, (256,))

losses = [
    float(train_step(model, optimizer, jax.random.key(i), (x_data, y_data)))
    for i in range(5)
]
print(f"linear-fit loss: initial {losses[0]:.4f} -> after a few steps {losses[-1]:.4f}")
print("Environment OK")

## 1. Pointers first (self-paced reading)

If JAX is new to you, skim these two references; everything below covers
only the subset that actually appears in our notebooks.

1. [Official JAX quickstart](https://docs.jax.dev/en/latest/quickstart.html)
   -- the canonical tour of arrays, PRNG keys, `jit`, `grad`, `vmap`.
2. [UvA Deep Learning Course: Introduction to JAX+Flax](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/JAX/tutorial2/Introduction_to_JAX.html)
   -- written explicitly for PyTorch users, with side-by-side comparisons.

## 2. The six things you need (📦)

Each subsection below is a single demonstration cell.

### 2.1 Arrays are immutable

`jax.numpy` (imported as `jnp`) mirrors numpy, but arrays are **immutable**:
there is no in-place mutation.
Updates are functional -- `x.at[idx].set(v)` returns a *new* array.
You will hit this the moment you try a torch (or standard-numpy) style in-place edit.
Our sampler loops sidestep this by rebinding `x = x + ...` each step;
the `.at` idiom above covers the rare cases where you truly need an
indexed update.

In [ ]:
x = jnp.arange(5.0)
try:
    x[0] = 10.0
except TypeError as e:
    print("in-place assignment fails:", str(e).splitlines()[0])

x_new = x.at[0].set(10.0)  # functional update
print("x     =", x)
print("x_new =", x_new)

### 2.2 Broadcasting per-sample scalars

The most common mistake in Notebook 1's Exercise 1 is a broadcasting
one: with a batch `x: (B, 2)` and per-sample times `t: (B,)`, the
product `alpha(t) * x` raises a shape error -- or, worse, silently
misbroadcasts when the shapes happen to line up.
The fix is to add a trailing axis: `alpha(t)[:, None] * x` has shape
`(B, 1) * (B, 2) -> (B, 2)`.
Every exercise in Notebook 1 is followed by a shape-assert cell that
catches this for you.

In [ ]:
B = 4
x = jnp.ones((B, 2))       # batch of 2D points
t = jnp.linspace(0.1, 0.9, B)  # one time per sample, shape (B,)
alpha = lambda t: t        # stand-in schedule coefficient

try:
    bad = alpha(t) * x     # (B,) * (B, 2): error
except (TypeError, ValueError) as e:
    print("shape error:", e)

good = alpha(t)[:, None] * x   # (B, 1) * (B, 2) -> (B, 2)
print("fixed shape:", good.shape)

### 2.3 Explicit PRNG keys

PRNG (pseudo-random number generator) state is **explicit** in JAX.
There is no global seed: every random call consumes a key, and you create
new ones with `jax.random.split`.
The same key always produces the same numbers.

In the later notebooks the exercise skeletons hand you **pre-split keys**
(`key_t`, `key_eps`), so you only ever consume keys.

In [ ]:
key = jax.random.key(0)
key_t, key_eps = jax.random.split(key)

print("normal draw:", jax.random.normal(key_eps, (3,)))
print("same key   :", jax.random.normal(key_eps, (3,)))  # identical -- keys are deterministic

### 2.4 `jit`, `grad`, `vmap`

- `jax.jit` just-in-time-compiles a function for speed (it compiles on
  first call). To make it work, avoid Python side effects and shapes that
  depend on array values inside a jitted function.
- `jax.grad` and `jax.value_and_grad` take gradients of scalar-valued functions.
- `jax.vmap` vectorizes a single-sample function over a batch axis.

In [ ]:
f = lambda x: jnp.sum(x**2)

f_fast = jax.jit(f)
print("jit  :", f_fast(jnp.arange(3.0)))

print("grad :", jax.grad(f)(jnp.arange(3.0)))  # 2x
value, grad = jax.value_and_grad(f)(jnp.arange(3.0))
print("value_and_grad :", value, grad)

g = lambda x: x @ x   # defined for a single vector (2,)
print("vmap :", jax.vmap(g)(jnp.ones((4, 2))))  # applied to a batch (4, 2)

### 2.5 flax.nnx modules and the canonical train step

`flax.nnx` is Flax's object-oriented neural-network API: an `nnx.Module`
holds its parameters as attributes (like `torch.nn.Module`), and
`rngs = nnx.Rngs(0)` supplies initialization randomness.
Reference doc: [flax nnx basics](https://flax.readthedocs.io/en/latest/nnx_basics.html).

The `LinearModel` from the install check is the whole pattern:
`nnx.Param` attributes in `__init__`, plain math in `__call__`.

In [ ]:
model_demo = LinearModel(rngs=nnx.Rngs(0))
print("params:", nnx.state(model_demo, nnx.Param))
print("model_demo(1.0) =", model_demo(jnp.array(1.0)))

Training uses **one** scaffold, reused, with minor variations, in every
notebook of this tutorial:

```python
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

@nnx.jit
def train_step(model, optimizer, key, batch):
    def loss_fn(model):
        return ...  # the exercise usually lives here
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads=grads, model=model)
    return loss
```

Two things:

- Use `nnx.jit` and `nnx.value_and_grad` rather than the bare `jax.*`
  versions; they know how to handle module state.
- `optimizer.update(grads=grads, model=model)` mutates the model **in
  place**. Unlike PyTorch, there is no separate `loss.backward()`
  followed by `optimizer.step()`.

### 2.6 Pytrees, in one sentence

A *pytree* is any nested container (dict/list/tuple) of arrays, and
`jax.tree.map` applies a function to every leaf -- that is all you need to
read our code.

In [ ]:
params = {"a": jnp.ones((2, 3)), "inner": {"b": jnp.zeros(4)}}
print(jax.tree.map(lambda leaf: leaf.shape, params))

## 3. ✏️ Micro-exercise: linear fit (optional)

Fit `y = a·x + b` by gradient descent to confirm the whole toolchain
(jit, grad, optimizer, PRNG) works end to end.
The model and data are provided 📦; you fill in only the body of
`loss_fn` inside the scaffold from section 2.5.

The loss is the MSE (mean squared error):

$$ \mathcal{L}(a, b) = \frac{1}{N} \sum_{i=1}^{N} \big( a\, x_i + b - y_i \big)^2 $$

In [ ]:
# 📦 Data (truth: a = 2.0, b = -0.7); the model is `LinearModel` from 2.5.
A_TRUE, B_TRUE = 2.0, -0.7
key_x, key_noise = jax.random.split(jax.random.key(1))
x_data = jax.random.uniform(key_x, (256,), minval=-1.0, maxval=1.0)
y_data = A_TRUE * x_data + B_TRUE + 0.1 * jax.random.normal(key_noise, (256,))

In [ ]:
# ✏️ Fill in the MSE loss. Shapes: batch = (x, y) with x: (N,), y: (N,);
# loss_fn returns a scalar ().
model = LinearModel(rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adam(1e-1), wrt=nnx.Param)


@nnx.jit
def train_step(model, optimizer, key, batch):
    # `key` is unused here -- this fit is full-batch and deterministic; it is
    # in the signature because every later notebook's loss_fn draws t and
    # ε from it.
    def loss_fn(model):
        """model -> scalar MSE loss over the full batch."""
        x, y = batch
        raise NotImplementedError  # YOUR CODE HERE

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads=grads, model=model)
    return loss

In [ ]:
# 📦 Test cell: shape checks + a short training run.
loss0 = train_step(model, optimizer, jax.random.key(0), (x_data, y_data))
assert jnp.shape(loss0) == (), f"loss must be a scalar, got shape {jnp.shape(loss0)}"
assert jnp.isfinite(loss0), "loss is not finite -- check the loss_fn body"

for step in range(300):
    loss = train_step(model, optimizer, jax.random.key(step), (x_data, y_data))

a_fit, b_fit = float(model.a), float(model.b)
print(f"recovered a = {a_fit:+.3f}   (truth {A_TRUE:+.3f})")
print(f"recovered b = {b_fit:+.3f}   (truth {B_TRUE:+.3f})")
print(f"final loss  = {float(loss):.4f}")
assert abs(a_fit - A_TRUE) < 0.1 and abs(b_fit - B_TRUE) < 0.1, (
    "fit did not converge to the truth -- check the loss_fn body"
)
print("Micro-exercise passed -- the full toolchain works.")

That's everything.
See you in Notebook 1 -- and if the install check above did not end with
`Environment OK`, sort that out (or ask us) *before* the first session.